# 03 - Gelistirilmis NumPy Modelleri

Bu notebook, laboratuvar temel modelinin uzerine iki farkli NumPy mimarisi kurarak performans degisimini inceler.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_utils import RANDOM_SEED, prepare_datasets, set_global_seed
from src.metrics import assess_fit_from_history, evaluate_classification
from src.numpy_models import NumpyMLPClassifier
from src.visualization import plot_confusion_matrix_figure, plot_multi_model_history


In [ ]:
set_global_seed(RANDOM_SEED)
DATA_PATH = PROJECT_ROOT / 'data' / 'heart_failure_clinical_records_dataset.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
split = prepare_datasets(DATA_PATH, random_state=RANDOM_SEED)

model_configs = {
    'NumPy Baseline': {'layer_dims': [split.X_train.shape[1], 8, 1], 'learning_rate': 0.05, 'epochs': 400, 'batch_size': None, 'l2_lambda': 0.0, 'initialization': 'lab'},
    'NumPy Model A': {'layer_dims': [split.X_train.shape[1], 16, 8, 1], 'learning_rate': 0.01, 'epochs': 600, 'batch_size': None, 'l2_lambda': 0.0, 'initialization': 'xavier'},
    'NumPy Model B': {'layer_dims': [split.X_train.shape[1], 32, 16, 1], 'learning_rate': 0.005, 'epochs': 300, 'batch_size': 32, 'l2_lambda': 0.001, 'initialization': 'xavier'},
}

histories = {}
comparison_rows = []
test_predictions_map = {}

for model_name, config in model_configs.items():
    model = NumpyMLPClassifier(seed=RANDOM_SEED, **config)
    histories[model_name] = model.fit(split.X_train, split.y_train, split.X_val, split.y_val)
    train_pred = model.predict(split.X_train)
    val_pred = model.predict(split.X_val)
    test_pred = model.predict(split.X_test)
    test_predictions_map[model_name] = test_pred
    train_metrics = evaluate_classification(split.y_train, train_pred, model.predict_proba(split.X_train))
    val_metrics = evaluate_classification(split.y_val, val_pred, model.predict_proba(split.X_val))
    test_metrics = evaluate_classification(split.y_test, test_pred, model.predict_proba(split.X_test))
    comparison_rows.append({
        'model_name': model_name,
        'architecture': ' -> '.join(str(value) for value in config['layer_dims']),
        'epochs': config['epochs'],
        'train_accuracy': round(train_metrics['accuracy'], 4),
        'validation_accuracy': round(val_metrics['accuracy'], 4),
        'test_accuracy': round(test_metrics['accuracy'], 4),
        'precision': round(test_metrics['precision'], 4),
        'recall': round(test_metrics['recall'], 4),
        'specificity': round(test_metrics['specificity'], 4),
        'f1_score': round(test_metrics['f1_score'], 4),
        'fit_comment': assess_fit_from_history(histories[model_name]),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(by=['validation_accuracy', 'test_accuracy', 'f1_score'], ascending=False)
comparison_df

In [ ]:
plot_multi_model_history(histories, FIGURE_DIR / 'numpy_models_validation_loss.png', 'val_loss', 'Improved NumPy Models Validation Loss')
plot_multi_model_history(histories, FIGURE_DIR / 'numpy_models_validation_accuracy.png', 'val_accuracy', 'Improved NumPy Models Validation Accuracy')
plot_confusion_matrix_figure(split.y_test, test_predictions_map['NumPy Model A'], FIGURE_DIR / 'numpy_model_a_confusion_matrix.png', 'NumPy Model A Confusion Matrix')
plot_confusion_matrix_figure(split.y_test, test_predictions_map['NumPy Model B'], FIGURE_DIR / 'numpy_model_b_confusion_matrix.png', 'NumPy Model B Confusion Matrix')

Yorum: Ilk denemelerde daha derin modellerin egitim dogrulugu hizla artarken dogrulama dogrulugu ayni oranda ilerlememektedir. Bu nedenle model secimi sadece train accuracy ile yapilmamali, validation ve test tarafindaki genel davranis birlikte incelenmelidir.

In [ ]:
best_model_row = comparison_df.iloc[0]
best_model_row

Sonuc: Bu deney kurulumunda en iyi NumPy modeli `NumPy Model A` olarak secilmistir. Bunun nedeni, temel modele gore hem dogrulama dogrulugunu hem de test F1-score degerini yukari cekmesidir. Model B daha genis ve L2 duzenlilestirmeli olmasina ragmen ayni veri setinde daha dengeli ama daha dusuk bir sonuc vermistir.